# 第1章 Unix/Linux 与科研计算环境

这个 notebook 不只是骨架，而是一个可运行的最小练习流。我们会在一个临时项目目录里演示路径、目录组织、基础命令、搜索、权限和归档这些最常见的 Linux 科研操作。

## 学习目标

- 理解当前工作目录、绝对路径和相对路径的差别
- 看清科研项目目录结构为什么重要
- 用最小例子演示 `pwd`、`ls`、`mkdir`、`cp`、`mv`、`rm`、`find`
- 认识一条命令由命令名、选项和参数组成，并学会先看 `--help`
- 用 `rg` 或替代方法做内容搜索
- 理解磁盘占用、执行权限和简单归档操作在科研工作流中的作用

In [ ]:
from __future__ import annotations

import platform
import shutil
import subprocess
import tempfile
from pathlib import Path


def run_shell(command: str, cwd: Path | None = None) -> str:
    completed = subprocess.run(
        ['bash', '-lc', command],
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        check=True,
    )
    return completed.stdout.strip()


tmpdir = tempfile.TemporaryDirectory(prefix='aifor_linux_demo_')
PROJECT_ROOT = Path(tmpdir.name) / 'gaia_project'
print('Python version:', platform.python_version())
print('Notebook working directory:', Path.cwd())
print('Temporary project root:', PROJECT_ROOT)


## 先建立一个最小科研目录

真实项目里，目录结构本身就是工作流的一部分。这里我们先建一个非常小的项目骨架。

In [ ]:
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for relative in ['raw', 'intermediate', 'scripts', 'logs', 'results']:
    (PROJECT_ROOT / relative).mkdir(exist_ok=True)

(PROJECT_ROOT / 'raw' / 'night1_target042.fits').write_text('fake fits placeholder\n', encoding='utf-8')
(PROJECT_ROOT / 'raw' / 'night1_target108.fits').write_text('fake fits placeholder\n', encoding='utf-8')
(PROJECT_ROOT / 'logs' / 'night1.log').write_text(
    'target_042 exposure_ok\n'
    'target_108 cloud_warning\n',
    encoding='utf-8',
)

print(run_shell('pwd', cwd=PROJECT_ROOT))
print(run_shell('ls -1', cwd=PROJECT_ROOT))


## 当前目录、绝对路径和相对路径

`pwd` 回答的是“我现在在哪”；绝对路径从根开始写，相对路径从当前目录出发写。

In [ ]:
current_dir = run_shell('pwd', cwd=PROJECT_ROOT)
relative_example = Path('raw/night1_target042.fits')
absolute_example = (PROJECT_ROOT / relative_example).resolve()
print('pwd ->', current_dir)
print('relative path ->', relative_example)
print('absolute path ->', absolute_example)


## 命令的基本结构与帮助系统

一条命令通常由命令名、选项和参数组成。遇到不确定的地方，先看 `--help`。

In [ ]:
command = 'ls -lh raw'
parts = {'command_name': 'ls', 'options': ['-l', '-h'], 'argument': 'raw'}
print('command ->', command)
print('parts ->', parts)
print(run_shell('ls --help | head -n 3', cwd=PROJECT_ROOT))


## 文件管理：复制、重命名、删除

下面演示的是最基础但也最常用的一组文件操作。

In [ ]:
run_shell('cp raw/night1_target042.fits intermediate/target042_copy.fits', cwd=PROJECT_ROOT)
run_shell('mv intermediate/target042_copy.fits intermediate/target042_reduced.fits', cwd=PROJECT_ROOT)
(PROJECT_ROOT / 'results' / 'quick_note.txt').write_text('temporary note\n', encoding='utf-8')
before_rm = run_shell('ls -1 results', cwd=PROJECT_ROOT)
run_shell('rm results/quick_note.txt', cwd=PROJECT_ROOT)
after_rm = run_shell('ls -1 results || true', cwd=PROJECT_ROOT)
print('intermediate contents:')
print(run_shell('ls -1 intermediate', cwd=PROJECT_ROOT))
print('results before rm:', before_rm)
print('results after rm:', after_rm if after_rm else '[empty]')


## 搜索：文件在哪，内容里有什么

`find` 适合找路径，`rg` 更适合找文本内容。若环境里没有 `rg`，就退回 `grep`。

In [ ]:
fits_listing = run_shell("find raw -name '*.fits' | sort", cwd=PROJECT_ROOT)
search_tool = 'rg' if shutil.which('rg') else 'grep -R'
if search_tool == 'rg':
    keyword_hits = run_shell("rg 'target_042' logs/", cwd=PROJECT_ROOT)
else:
    keyword_hits = run_shell("grep -R 'target_042' logs/", cwd=PROJECT_ROOT)

print('find result:')
print(fits_listing)
print('content search using', search_tool)
print(keyword_hits)


## 目录大小与磁盘空间

`du` 回答“这个目录占了多少空间”，`df` 回答“这个文件系统还剩多少空间”。两者经常被混淆。

In [ ]:
print('directory usage:')
print(run_shell('du -sh raw results', cwd=PROJECT_ROOT))
print('filesystem free space:')
print(run_shell('df -h . | tail -n 1', cwd=PROJECT_ROOT))


## 重定向与记录

科研里很多终端输出都值得存档。下面演示把目录快照写进一个文本记录。

In [ ]:
run_shell('ls -lh raw > logs/raw_snapshot.txt', cwd=PROJECT_ROOT)
print((PROJECT_ROOT / 'logs' / 'raw_snapshot.txt').read_text(encoding='utf-8'))


## 权限：脚本为什么要加执行位

对脚本而言，执行权限决定它能不能被直接运行。

In [ ]:
script_path = PROJECT_ROOT / 'scripts' / 'run_pipeline.sh'
script_path.write_text('#!/usr/bin/env bash\necho pipeline-ready\n', encoding='utf-8')
before_perm = run_shell('ls -l scripts/run_pipeline.sh', cwd=PROJECT_ROOT)
run_shell('chmod u+x scripts/run_pipeline.sh', cwd=PROJECT_ROOT)
after_perm = run_shell('ls -l scripts/run_pipeline.sh', cwd=PROJECT_ROOT)
script_output = run_shell('./scripts/run_pipeline.sh', cwd=PROJECT_ROOT)
print('before chmod:', before_perm)
print('after chmod:', after_perm)
print('script run ->', script_output)


## 归档：把阶段性结果打包

科研目录经常需要阶段性打包，方便备份、传输和归档。

In [ ]:
run_shell('tar -czf results/night1_bundle.tar.gz raw logs scripts', cwd=PROJECT_ROOT)
archive_listing = run_shell('tar -tzf results/night1_bundle.tar.gz | head', cwd=PROJECT_ROOT)
print('archive contents preview:')
print(archive_listing)


## 小结

这个最小练习流说明：Linux 命令不是零散技巧，而是一条从目录组织、文件管理、搜索、记录到归档的科研工作流接口。真正重要的不是死记命令，而是理解它们如何共同支撑一个可复现项目。

In [ ]:
snapshot = {
    'project_root': str(PROJECT_ROOT),
    'raw_files': run_shell("find raw -name '*.fits' | wc -l", cwd=PROJECT_ROOT),
    'search_tool': search_tool,
    'command_parts_ok': parts['command_name'] == 'ls',
    'disk_usage_checked': True,
    'archive_exists': (PROJECT_ROOT / 'results' / 'night1_bundle.tar.gz').exists(),
    'python_version': platform.python_version(),
}

print('Linux chapter snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

本章的记录重点是项目根目录、只读检查、写操作前预览、日志文件和命令入口。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch01_evidence_record.md'

archive_path = PROJECT_ROOT / 'results' / 'night1_bundle.tar.gz'
log_path = PROJECT_ROOT / 'logs' / 'raw_snapshot.txt'
project_files_preview = run_shell('find . -maxdepth 2 -type f | sort', cwd=PROJECT_ROOT)
python_location = run_shell('command -v python || command -v python3', cwd=PROJECT_ROOT)
preview_note = f"before `rm`, results contained: {before_rm!r}"

record_text = f"""# Evidence Record - Ch01 Unix/Linux Scientific Computing

## Record Type

- Record type: command / reproducibility evidence

## 1. Input

- Data / file path, preferably relative to project root: temporary project root `{PROJECT_ROOT}` with `raw/`, `logs/`, `scripts/`, `results/`
- Data source or generation method: notebook-created teaching project with fake FITS placeholders and a small log file
- Script / notebook: `ch01_unix_linux_scientific_computing.ipynb`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: notebook cells call shell commands through `run_shell()`
- Key parameters: project directories `raw`, `intermediate`, `scripts`, `logs`, `results`; archive `results/night1_bundle.tar.gz`
- Random seed, if relevant: not applicable
- Output directory: notebook-local `results/` for this Evidence Record; temporary project `results/` for demo archive

## 3. Output

- Output file(s): `{evidence_path}`; demo archive `{archive_path}`; demo log `{log_path}`
- Short result summary: raw FITS placeholders `{snapshot['raw_files']}`; search tool `{search_tool}`; archive exists `{archive_path.exists()}`
- Generated by which script/notebook: `ch01_unix_linux_scientific_computing.ipynb`

## 4. Check

- Check performed: `pwd`, `ls`, `find`, content search, `du`, `df`, log redirection, permission change, archive listing
- Evidence from the check: `find` output `{fits_listing.splitlines()}`; keyword hit `{keyword_hits}`; archive preview `{archive_listing.splitlines()[:3]}`

## 5. Limit

- Known limitation: all project files live in a temporary directory and are teaching placeholders, not real astronomical data products
- Selection / quality / uncertainty issue, if relevant: fake FITS files contain no science image data or metadata

## 6. Reuse in ML

- Sample / feature / label: not a model dataset yet
- Uncertainty / quality flag: not applicable
- Preprocessing record: documents command-line project layout, logging, search, and archive habits
- Reproducibility evidence: records project root, command checks, preview-before-delete habit, log file, and Python version

## Ch01 Fields

- project root: `{PROJECT_ROOT}`
- key read-only checks: `pwd`, `ls --help`, `find`, `{search_tool}`, `du -sh`, `df -h`, `tar -tzf`
- preview before write: `{preview_note}`; raw directory snapshot written after listing command was explicit
- log file: `logs/raw_snapshot.txt`
- script or command entry: shell commands executed through `run_shell()`; demo script `scripts/run_pipeline.sh`
- python path/version check: `{python_location}`; Python `{platform.python_version()}`

## File Preview

```text
{project_files_preview}
```
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
